In [29]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import matplotlib.pyplot as plt
import evaluate

Loading and preprocessing the dataset

In [30]:
dataset = pd.read_csv("../datasets/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")

# Normalize column names (lowercase, strip spaces)
dataset.columns = dataset.columns.str.strip().str.lower()

# Check required columns exist
required_cols = {"instruction", "intent"}
if not required_cols.issubset(dataset.columns):
    raise ValueError(f"Dataset must contain columns: {required_cols}")

# Remove rows with missing values
dataset.dropna(subset=["instruction", "intent"], inplace=True)

print(f"✅ Dataset loaded successfully: {len(dataset)} samples")

features = dataset["instruction"]
labels = dataset["intent"]

# Enconfing the labels for use by the LLM
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

# Save label mapping
label_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
print("✅ Label mapping:", label_mapping)

✅ Dataset loaded successfully: 26872 samples
✅ Label mapping: {'cancel_order': 0, 'change_order': 1, 'change_shipping_address': 2, 'check_cancellation_fee': 3, 'check_invoice': 4, 'check_payment_methods': 5, 'check_refund_policy': 6, 'complaint': 7, 'contact_customer_service': 8, 'contact_human_agent': 9, 'create_account': 10, 'delete_account': 11, 'delivery_options': 12, 'delivery_period': 13, 'edit_account': 14, 'get_invoice': 15, 'get_refund': 16, 'newsletter_subscription': 17, 'payment_issue': 18, 'place_order': 19, 'recover_password': 20, 'registration_problems': 21, 'review': 22, 'set_up_shipping_address': 23, 'switch_account': 24, 'track_order': 25, 'track_refund': 26}


Splitting between training and testing data

In [31]:
feature_train, feature_test, label_train, label_test = train_test_split(
    features,
    encoded_labels,
    test_size=0.2,
    stratify=encoded_labels,
    random_state=42
)

Tokenizing the dataset and preparing the model for training

In [32]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
train_encodings = tokenizer(feature_train.tolist(), truncation=True, padding=True)
test_encodings = tokenizer(feature_test.tolist(), truncation=True, padding=True)

# Making torch datasets
class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = IntentDataset(train_encodings, label_train.tolist())
test_dataset = IntentDataset(test_encodings, label_test.tolist())

# Initializing the model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_mapping)
)

# Training configurations
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_steps=0,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

Evaluation metrics

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    }